In [ ]:
-- =============================================================================
-- SLEEPER TRADES PIPELINE
-- Dynasty-Aware Multi-Season Trade Analysis
-- 
-- Key Features:
-- - Multi-season player point tracking
-- - Draft pick career point tracking
-- - Trade completeness detection (all picks realized)
-- - Dynasty vs redraft league support
-- - Trade impact calculation across multiple time horizons
-- =============================================================================

In [ ]:
-- ---------- STAGING ----------

In [ ]:
-- Base trade transactions with league context
CREATE OR REPLACE MATERIALIZED VIEW stg_trade_transactions AS
SELECT
  t.league_id,
  li.season,
  t.transaction_id,
  t.status,
  t.creator,
  t.adds,
  t.drops,
  t.roster_ids,
  t.draft_picks,
  t.leg AS week,
  t.created,
  to_timestamp(t.created/1000.0) AS event_ts,
  lower(regexp_replace(coalesce(li.name,'unknown'), '[^a-zA-Z0-9]+', '_')) AS cluster_key,
  li.name AS cluster_name
FROM workspace.sleeper_raw.sleeper_transactions_snapshot t
JOIN workspace.sleeper_raw.sleeper_league_info_snapshot li USING (league_id)
WHERE t.type = 'trade' AND t.status = 'complete';

In [ ]:
-- Trade metadata with league type
CREATE OR REPLACE MATERIALIZED VIEW dim_trade_metadata AS
SELECT
  t.league_id,
  t.transaction_id,
  t.season AS trade_season,
  t.week AS trade_week,
  t.event_ts AS trade_date,
  t.cluster_key,
  t.cluster_name,
  lc.league_type,
  lc.season_start AS league_start_season,
  lc.season_end AS league_end_season
FROM stg_trade_transactions t
LEFT JOIN workspace.sleeper_core.dim_league_config lc ON t.league_id = lc.league_id;

In [ ]:
-- ---------- PLAYER ASSETS ----------

In [ ]:
-- Player assets in trades (who got which players)
CREATE OR REPLACE MATERIALIZED VIEW fact_trade_player_assets AS
WITH tx AS (SELECT * FROM stg_trade_transactions),
adds_raw AS (
  SELECT
    league_id,
    season,
    transaction_id,
    week,
    event_ts,
    cluster_key,
    cluster_name,
    explode(transform(map_entries(adds), e -> 
      named_struct('player_id', e.key, 'roster_id', e.value)
    )) AS add_struct
  FROM tx
),
adds AS (
  SELECT
    league_id,
    season,
    transaction_id,
    week,
    event_ts,
    cluster_key,
    cluster_name,
    add_struct.player_id,
    add_struct.roster_id AS side_roster_id,
    'incoming' AS direction
  FROM adds_raw
),
drops_raw AS (
  SELECT
    league_id,
    season,
    transaction_id,
    week,
    event_ts,
    cluster_key,
    cluster_name,
    explode(transform(map_entries(drops), e -> 
      named_struct('player_id', e.key, 'roster_id', e.value)
    )) AS drop_struct
  FROM tx
),
drops AS (
  SELECT
    league_id,
    season,
    transaction_id,
    week,
    event_ts,
    cluster_key,
    cluster_name,
    drop_struct.player_id,
    drop_struct.roster_id AS side_roster_id,
    'outgoing' AS direction
  FROM drops_raw
)
SELECT * FROM adds
UNION ALL
SELECT * FROM drops;

In [ ]:
-- Multi-season player points after trade
-- Uses cluster_key for cross-season tracking (league_id changes each season in Sleeper)
-- Counts ALL career points scored by the player regardless of subsequent ownership
CREATE OR REPLACE MATERIALIZED VIEW fact_trade_player_points_multi_season AS
WITH incoming_players AS (
  SELECT 
    league_id,
    transaction_id,
    side_roster_id,
    player_id,
    season AS trade_season,
    week AS trade_week,
    cluster_key
  FROM fact_trade_player_assets
  WHERE direction = 'incoming'
),
-- Get all points scored by acquired players after the trade
points_after_trade AS (
  SELECT
    ip.league_id,
    ip.transaction_id,
    ip.side_roster_id,
    ip.player_id,
    ip.trade_season,
    ip.trade_week,
    pw.season AS scoring_season,
    pw.week AS scoring_week,
    pw.points,
    -- Calculate seasons since trade
    CAST(pw.season AS INT) - CAST(ip.trade_season AS INT) AS seasons_after_trade,
    -- Calculate weeks since trade (approximate, doesn't account for season boundaries)
    CASE 
      WHEN pw.season = ip.trade_season THEN pw.week - ip.trade_week
      ELSE NULL  -- Cross-season week calculations are complex, use season delta instead
    END AS weeks_after_trade
  FROM incoming_players ip
  JOIN workspace.sleeper_core.fact_player_week_enriched pw
    ON ip.cluster_key = pw.cluster_key
    AND ip.player_id = pw.player_id
  WHERE 
    -- Only count points after the trade
    (CAST(pw.season AS INT) > CAST(ip.trade_season AS INT))
    OR (pw.season = ip.trade_season AND pw.week > ip.trade_week)
)
SELECT
  league_id,
  transaction_id,
  side_roster_id,
  player_id,
  trade_season,
  trade_week,
  scoring_season,
  scoring_week,
  points,
  seasons_after_trade,
  weeks_after_trade
FROM points_after_trade;

In [ ]:
-- ---------- DRAFT PICK ASSETS ----------

In [ ]:
-- Draft picks included in trades with realization tracking
-- Key insight: roster_id in traded picks = ORIGINAL owner of the pick (whose pick it naturally is)
CREATE OR REPLACE MATERIALIZED VIEW fact_trade_pick_assets AS
WITH tx AS (
  SELECT league_id, transaction_id, season AS trade_season, draft_picks
  FROM stg_trade_transactions
),
parsed AS (
  SELECT 
    league_id,
    transaction_id,
    trade_season,
    transform(
      CAST(draft_picks AS ARRAY<STRING>),
      p -> from_json(p, 'MAP<STRING,STRING>')
    ) AS picks
  FROM tx
  WHERE draft_picks IS NOT NULL AND size(CAST(draft_picks AS ARRAY<STRING>)) > 0
),
exploded AS (
  SELECT 
    league_id,
    transaction_id,
    trade_season,
    explode(picks) AS pick_map
  FROM parsed
),
normalized AS (
  SELECT
    league_id,
    transaction_id,
    trade_season,
    CAST(pick_map['round'] AS INT) AS round,
    CAST(pick_map['roster_id'] AS INT) AS original_owner_id,  -- This is the ORIGINAL owner
    CAST(pick_map['previous_owner_id'] AS INT) AS from_roster_id,  -- Who's trading it away
    CAST(pick_map['owner_id'] AS INT) AS to_roster_id,  -- Who's receiving it
    -- Pick realization season (when the pick will be used)
    COALESCE(
      pick_map['season'],
      CAST(CAST(trade_season AS INT) + 1 AS STRING)
    ) AS pick_season
  FROM exploded
)
SELECT
  league_id,
  transaction_id,
  trade_season,
  round,
  original_owner_id,  -- Track the original owner (whose pick it naturally belongs to)
  from_roster_id,  -- Who traded it in THIS transaction
  to_roster_id AS side_roster_id,  -- Who received it in THIS transaction
  pick_season,
  CAST(pick_season AS INT) - CAST(trade_season AS INT) AS seasons_until_realization
FROM normalized;

In [ ]:
-- =============================================================================
-- bridge_trade_pick_to_player: Maps traded draft picks to drafted players
-- =============================================================================
-- PURPOSE:
--   Matches draft picks included in trades to the players actually drafted with
--   those picks. Enables calculation of pick value by tracking career points of
--   players drafted with traded picks. Handles complex scenario where picks are
--   traded multiple times before being used.
--
-- GRAIN:
--   One row per (transaction, pick) - same pick appears multiple times if re-traded
--
-- COLUMNS:
--   league_id:            League where trade occurred
--   transaction_id:       Trade transaction identifier
--   side_roster_id:       Roster that received the pick in THIS trade
--   trade_season:         Season when trade occurred
--   pick_season:          Season when pick will be/was used in draft
--   round:                Draft round (1-4 typical for dynasty)
--   original_owner_id:    Roster whose natural pick this is (determines pick_no)
--   from_roster_id:       Roster trading the pick away in THIS transaction
--   pick_no:              Calculated pick number based on original owner's draft slot
--   player_id:            Player drafted with this pick (NULL if not yet drafted)
--   draft_id:             Draft where pick was used (NULL if not yet drafted)
--   drafter_roster_id:    Roster that actually used the pick (may differ from side_roster_id if re-traded)
--   is_realized:          TRUE if pick has been used to draft a player
--
-- KEY BUSINESS LOGIC:
--   1. Extracts traded picks with original_owner_id (whose pick it naturally is)
--   2. Calculates pick_no based on original owner's draft slot + round + draft type:
--      - Snake drafts: Odd rounds normal order, even rounds reverse
--      - Linear drafts: Same order every round
--      Example: 12-team snake, original_owner has slot 3
--        - Round 1 (odd): pick_no = 3
--        - Round 2 (even): pick_no = 22 (12*2 - 3 + 1)
--   3. Matches to actual draft by (cluster_key, season, pick_no)
--   4. Does NOT verify final drafter matches trade recipient (enables re-traded picks)
--
-- IMPORTANT NOTES:
--   - Pick identity is determined by original_owner_id, not the current holder
--   - drafter_roster_id may differ from side_roster_id when picks are re-traded
--   - Uses cluster_key for cross-season matching (Sleeper creates new league_ids annually)
--
-- DATA QUALITY:
--   - is_realized should eventually become TRUE for all past-season picks
--   - pick_no can be NULL if original_owner's draft slot is unknown
--   - Same pick appears multiple times if traded more than once
--   - Use bridge_trade_pick_unique for deduplicated pick view
--
-- DEPENDENCIES:
--   - fact_trade_pick_assets: Extracted traded picks from transactions
--   - sleeper_drafts_snapshot: Draft metadata
--   - sleeper_draft_slot_to_roster_snapshot: Draft slot assignments  
--   - sleeper_draft_picks_snapshot: Actual draft results
--   - sleeper_league_info_snapshot: League metadata for cluster_key
-- =============================================================================
CREATE OR REPLACE MATERIALIZED VIEW bridge_trade_pick_to_player AS
WITH trade_picks_with_cluster AS (
  SELECT
    pa.*,
    lower(regexp_replace(coalesce(li.name, 'unknown'), '[^a-zA-Z0-9]+', '_')) AS cluster_key
  FROM fact_trade_pick_assets pa
  JOIN workspace.sleeper_raw.sleeper_league_info_snapshot li ON pa.league_id = li.league_id
),
-- Get draft slot assignments: which roster_id had which slot in each draft
draft_slots AS (
  SELECT
    dr.draft_id,
    dr.league_id,
    dr.season,
    slot.slot,
    slot.roster_id,
    CAST(dr.settings['rounds'] AS INT) AS num_rounds,
    CAST(dr.settings['teams'] AS INT) AS num_teams,
    dr.type AS draft_type,
    lower(regexp_replace(coalesce(li.name, 'unknown'), '[^a-zA-Z0-9]+', '_')) AS cluster_key
  FROM workspace.sleeper_raw.sleeper_drafts_snapshot dr
  JOIN workspace.sleeper_raw.sleeper_draft_slot_to_roster_snapshot slot ON dr.draft_id = slot.draft_id
  JOIN workspace.sleeper_raw.sleeper_league_info_snapshot li ON dr.league_id = li.league_id
),
-- Calculate pick_no for each (draft, slot, round) based on draft type
draft_slot_pick_numbers AS (
  SELECT
    draft_id,
    cluster_key,
    season,
    slot,
    roster_id,
    round,
    num_teams,
    -- Calculate pick number based on draft type
    CASE
      WHEN draft_type = 'snake' THEN
        CASE
          WHEN round % 2 = 1 THEN (round - 1) * num_teams + slot  -- Odd rounds: slots 1,2,3... go picks 1,2,3...
          ELSE round * num_teams - slot + 1                        -- Even rounds: slots 1,2,3... go picks 24,23,22... (12 team)
        END
      ELSE (round - 1) * num_teams + slot  -- Linear: same order every round
    END AS pick_no
  FROM draft_slots
  CROSS JOIN (
    -- Generate all rounds for this draft (allows matching future picks before draft occurs)
    SELECT pos AS round 
    FROM explode(sequence(1, (SELECT MAX(num_rounds) FROM draft_slots))) AS t(pos)
  ) rounds
),
-- Get actual drafted players
drafted_players AS (
  SELECT
    dr.league_id,
    dr.season,
    CAST(dp.round AS INT) AS round,
    dp.roster_id AS drafter_roster_id,  -- Who actually made the pick (after all trades)
    dp.player_id,
    dp.pick_no,
    dp.draft_id,
    lower(regexp_replace(coalesce(li.name, 'unknown'), '[^a-zA-Z0-9]+', '_')) AS cluster_key
  FROM workspace.sleeper_raw.sleeper_draft_picks_snapshot dp
  JOIN workspace.sleeper_raw.sleeper_drafts_snapshot dr ON dp.draft_id = dr.draft_id
  JOIN workspace.sleeper_raw.sleeper_league_info_snapshot li ON dr.league_id = li.league_id
  WHERE dp.player_id IS NOT NULL  -- Filter out empty/skipped picks
)
SELECT
  tp.league_id,
  tp.transaction_id,
  tp.side_roster_id,
  tp.trade_season,
  tp.pick_season,
  tp.round,
  tp.original_owner_id,
  tp.from_roster_id,
  ds.pick_no,  -- The pick number that the original owner's slot had in this round
  drafted.player_id,
  drafted.draft_id,
  drafted.drafter_roster_id,
  -- Determine if pick has been realized
  CASE
    WHEN drafted.player_id IS NOT NULL THEN TRUE
    ELSE FALSE
  END AS is_realized
FROM trade_picks_with_cluster tp
-- Step 1: Find which slot the original owner had and calculate their pick number for this round
LEFT JOIN draft_slot_pick_numbers ds
  ON tp.cluster_key = ds.cluster_key
  AND tp.pick_season = ds.season
  AND tp.original_owner_id = ds.roster_id  -- Match on ORIGINAL owner to get correct pick_no
  AND tp.round = ds.round
-- Step 2: Match to the actual drafted player at that pick number
-- NOTE: We match solely on (cluster_key, season, pick_no) to handle re-traded picks correctly
LEFT JOIN drafted_players drafted
  ON ds.cluster_key = drafted.cluster_key
  AND ds.season = drafted.season
  AND ds.pick_no = drafted.pick_no;

In [ ]:
-- Career points from drafted players
-- Uses cluster_key for cross-season tracking (league_id changes each season in Sleeper)
-- Counts ALL career points scored by the player regardless of subsequent ownership
CREATE OR REPLACE MATERIALIZED VIEW fact_trade_pick_points_career AS
WITH realized_picks AS (
  SELECT
    bp.*,
    st.cluster_key
  FROM bridge_trade_pick_to_player bp
  JOIN stg_trade_transactions st ON bp.transaction_id = st.transaction_id
  WHERE bp.is_realized = TRUE
),
-- Get ALL points scored by the drafted player
career_points AS (
  SELECT
    rp.league_id,
    rp.transaction_id,
    rp.side_roster_id,
    rp.trade_season,
    rp.pick_season,
    rp.player_id,
    pw.season AS scoring_season,
    pw.week AS scoring_week,
    pw.points,
    CAST(pw.season AS INT) - CAST(rp.pick_season AS INT) AS seasons_after_draft
  FROM realized_picks rp
  JOIN workspace.sleeper_core.fact_player_week_enriched pw
    ON rp.cluster_key = pw.cluster_key  -- Match on cluster (not league_id)
    AND rp.player_id = pw.player_id      -- Match on player
  WHERE
    -- Only count points after the pick was used
    CAST(pw.season AS INT) >= CAST(rp.pick_season AS INT)
)
SELECT
  league_id,
  transaction_id,
  side_roster_id,
  player_id,
  trade_season,
  pick_season,
  scoring_season,
  scoring_week,
  points,
  seasons_after_draft
FROM career_points;

In [ ]:
-- Deduplicated draft picks view (one row per unique pick)
-- Use this when you want to count picks, not trade transactions
-- The bridge_trade_pick_to_player table has one row per trade, so the same pick
-- appears multiple times if it was traded more than once
CREATE OR REPLACE MATERIALIZED VIEW bridge_trade_pick_unique AS
WITH pick_with_cluster AS (
  SELECT
    bp.*,
    st.cluster_key,
    st.cluster_name
  FROM workspace.sleeper_trades.bridge_trade_pick_to_player bp
  JOIN workspace.sleeper_trades.stg_trade_transactions st
    ON bp.transaction_id = st.transaction_id
),
-- Get the most recent trade for each unique pick
ranked_picks AS (
  SELECT
    *,
    ROW_NUMBER() OVER (
      PARTITION BY cluster_key, pick_season, round, original_owner_id
      ORDER BY trade_season DESC, transaction_id DESC
    ) AS rn
  FROM pick_with_cluster
)
SELECT
  cluster_key,
  cluster_name,
  pick_season,
  round,
  original_owner_id,
  side_roster_id AS final_owner_id,  -- Who ended up with the pick
  transaction_id AS last_trade_id,
  player_id,
  pick_no,  -- The pick number based on original owner's slot
  draft_id,
  is_realized
FROM ranked_picks
WHERE rn = 1;

In [ ]:
-- ---------- TRADE COMPLETENESS ----------

In [ ]:
-- Determine if all picks in a trade have been realized
CREATE OR REPLACE MATERIALIZED VIEW dim_trade_completeness AS
WITH pick_status AS (
  SELECT
    league_id,
    transaction_id,
    COUNT(*) AS total_picks,
    SUM(CASE WHEN is_realized THEN 1 ELSE 0 END) AS realized_picks,
    MAX(pick_season) AS latest_pick_season
  FROM bridge_trade_pick_to_player
  GROUP BY league_id, transaction_id
),
trade_metadata AS (
  SELECT
    transaction_id,
    league_id,
    season AS trade_season
  FROM stg_trade_transactions
)
SELECT
  tm.transaction_id,
  tm.league_id,
  tm.trade_season,
  COALESCE(ps.total_picks, 0) AS total_picks,
  COALESCE(ps.realized_picks, 0) AS realized_picks,
  ps.latest_pick_season,
  -- Trade is complete if it has no picks OR all picks are realized
  CASE
    WHEN ps.total_picks IS NULL THEN TRUE
    WHEN ps.total_picks = ps.realized_picks THEN TRUE
    ELSE FALSE
  END AS is_complete
FROM trade_metadata tm
LEFT JOIN pick_status ps USING (league_id, transaction_id);

In [ ]:
-- ---------- TRADE IMPACT AGGREGATION ----------

In [ ]:
-- NEW: Trade impact by horizon (same season, 1 year, 2 years, career)
CREATE OR REPLACE MATERIALIZED VIEW agg_trade_impact_by_horizon AS
WITH player_points AS (
  SELECT
    league_id,
    transaction_id,
    side_roster_id,
    trade_season,
    seasons_after_trade,
    SUM(points) AS player_points
  FROM fact_trade_player_points_multi_season
  GROUP BY league_id, transaction_id, side_roster_id, trade_season, seasons_after_trade
),
pick_points AS (
  SELECT
    league_id,
    transaction_id,
    side_roster_id,
    trade_season,
    seasons_after_draft AS seasons_after_trade,
    SUM(points) AS pick_points
  FROM fact_trade_pick_points_career
  GROUP BY league_id, transaction_id, side_roster_id, trade_season, seasons_after_draft
),
combined AS (
  SELECT
    league_id,
    transaction_id,
    side_roster_id,
    trade_season,
    seasons_after_trade,
    COALESCE(player_points, 0) AS player_points,
    0 AS pick_points
  FROM player_points
  
  UNION ALL
  
  SELECT
    league_id,
    transaction_id,
    side_roster_id,
    trade_season,
    seasons_after_trade,
    0 AS player_points,
    COALESCE(pick_points, 0) AS pick_points
  FROM pick_points
),
aggregated AS (
  SELECT
    league_id,
    transaction_id,
    side_roster_id,
    trade_season,
    seasons_after_trade,
    SUM(player_points) AS player_points,
    SUM(pick_points) AS pick_points,
    SUM(player_points) + SUM(pick_points) AS total_points
  FROM combined
  GROUP BY league_id, transaction_id, side_roster_id, trade_season, seasons_after_trade
)
SELECT
  a.*,
  -- Add metadata
  dm.league_type,
  dm.cluster_key,
  dm.cluster_name
FROM aggregated a
LEFT JOIN dim_trade_metadata dm
  ON a.league_id = dm.league_id
  AND a.transaction_id = dm.transaction_id;

In [ ]:
-- NEW: Trade impact summary with multiple time horizons
CREATE OR REPLACE MATERIALIZED VIEW agg_trade_impact_summary AS
WITH horizons AS (
  SELECT
    league_id,
    transaction_id,
    side_roster_id,
    trade_season,
    league_type,
    cluster_key,
    cluster_name,
    -- Same season impact (seasons_after_trade = 0)
    SUM(CASE WHEN seasons_after_trade = 0 THEN total_points ELSE 0 END) AS same_season_points,
    -- Year 1 after trade
    SUM(CASE WHEN seasons_after_trade = 1 THEN total_points ELSE 0 END) AS year1_points,
    -- Year 2 after trade
    SUM(CASE WHEN seasons_after_trade = 2 THEN total_points ELSE 0 END) AS year2_points,
    -- Year 3+ after trade
    SUM(CASE WHEN seasons_after_trade >= 3 THEN total_points ELSE 0 END) AS year3plus_points,
    -- Total career impact
    SUM(total_points) AS career_points
  FROM agg_trade_impact_by_horizon
  GROUP BY league_id, transaction_id, side_roster_id, trade_season, league_type, cluster_key, cluster_name
)
SELECT
  h.*,
  tc.is_complete AS trade_is_complete,
  tc.total_picks,
  tc.realized_picks
FROM horizons h
LEFT JOIN dim_trade_completeness tc
  ON h.league_id = tc.league_id
  AND h.transaction_id = tc.transaction_id;

In [ ]:
-- NEW: Head-to-head trade comparison (who won the trade?)
CREATE OR REPLACE MATERIALIZED VIEW agg_trade_winners AS
WITH roster_pairs AS (
  SELECT DISTINCT
    league_id,
    transaction_id,
    side_roster_id
  FROM agg_trade_impact_summary
),
paired AS (
  SELECT
    a.league_id,
    a.transaction_id,
    a.side_roster_id AS roster_a,
    b.side_roster_id AS roster_b
  FROM roster_pairs a
  JOIN roster_pairs b
    ON a.league_id = b.league_id
    AND a.transaction_id = b.transaction_id
    AND a.side_roster_id < b.side_roster_id  -- Avoid duplicate pairs
),
with_scores AS (
  SELECT
    p.*,
    sa.career_points AS roster_a_points,
    sb.career_points AS roster_b_points,
    sa.trade_is_complete,
    sa.trade_season,
    sa.league_type,
    sa.cluster_key,
    sa.cluster_name
  FROM paired p
  JOIN agg_trade_impact_summary sa
    ON p.league_id = sa.league_id
    AND p.transaction_id = sa.transaction_id
    AND p.roster_a = sa.side_roster_id
  JOIN agg_trade_impact_summary sb
    ON p.league_id = sb.league_id
    AND p.transaction_id = sb.transaction_id
    AND p.roster_b = sb.side_roster_id
)
SELECT
  league_id,
  transaction_id,
  trade_season,
  league_type,
  cluster_key,
  cluster_name,
  roster_a,
  roster_b,
  roster_a_points,
  roster_b_points,
  roster_a_points - roster_b_points AS point_differential,
  CASE
    WHEN roster_a_points > roster_b_points THEN roster_a
    WHEN roster_b_points > roster_a_points THEN roster_b
    ELSE NULL  -- Tie
  END AS winner_roster_id,
  CASE
    WHEN roster_a_points < roster_b_points THEN roster_a
    WHEN roster_b_points < roster_a_points THEN roster_b
    ELSE NULL  -- Tie
  END AS loser_roster_id,
  ABS(roster_a_points - roster_b_points) AS trade_impact_magnitude,
  trade_is_complete
FROM with_scores;

In [ ]:
-- ---------- ENRICHED VIEWS WITH NAMES ----------

In [ ]:
-- Trade winners with manager names
CREATE OR REPLACE MATERIALIZED VIEW agg_trade_winners_enriched AS
SELECT
  tw.*,
  ma.manager_display_name AS roster_a_manager,
  mb.manager_display_name AS roster_b_manager,
  mw.manager_display_name AS winner_manager,
  ml.manager_display_name AS loser_manager
FROM agg_trade_winners tw
LEFT JOIN workspace.sleeper_core.dim_manager_roster_map ma
  ON tw.league_id = ma.league_id
  AND tw.roster_a = ma.roster_id
  AND tw.trade_season = ma.season
LEFT JOIN workspace.sleeper_core.dim_manager_roster_map mb
  ON tw.league_id = mb.league_id
  AND tw.roster_b = mb.roster_id
  AND tw.trade_season = mb.season
LEFT JOIN workspace.sleeper_core.dim_manager_roster_map mw
  ON tw.league_id = mw.league_id
  AND tw.winner_roster_id = mw.roster_id
  AND tw.trade_season = mw.season
LEFT JOIN workspace.sleeper_core.dim_manager_roster_map ml
  ON tw.league_id = ml.league_id
  AND tw.loser_roster_id = ml.roster_id
  AND tw.trade_season = ml.season;